In [1]:
from mpramnist.Fromel2025.dataset import FromelDataset
from mpramnist.Fromel2025.trainer import MaskedMSE, MaskedPearsonCorrCoef
from mpramnist.Fromel2025.trainer import LitModel_Fromel

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t
import mpramnist.target_transforms as t_t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import lightning.pytorch as L

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FromelDataset.HSPC_TARGETS

['State_1M',
 'State_2D',
 'State_3E',
 'State_4M',
 'State_5M',
 'State_6N',
 'State_7M']

In [3]:
def evaluate_with_reverse_complement(
    forward_loader: DataLoader,
    reverse_loader: DataLoader,
    trainer: L.Trainer,
    lit_model: LitModel_Fromel,
    name: str,
    num_outputs: int,
) -> torch.Tensor:
    """Predict on forward-strand and reverse-complement inputs separately,
    average the two predictions, and compute the masked Pearson correlation
    (per target) against the (shared) targets.
    """
    forward_preds = trainer.predict(lit_model, dataloaders=forward_loader)
    targets = torch.cat([batch["target"] for batch in forward_preds])
    forward_pred_values = torch.cat([batch["predicted"] for batch in forward_preds])

    reverse_preds = trainer.predict(lit_model, dataloaders=reverse_loader)
    reverse_pred_values = torch.cat([batch["predicted"] for batch in reverse_preds])

    mean_pred = torch.mean(torch.stack([forward_pred_values, reverse_pred_values]), dim=0)

    pearson_metric = MaskedPearsonCorrCoef(num_outputs=num_outputs)
    pearson = pearson_metric(mean_pred, targets)

    print("===========")
    print(name, " Pearson correlation")
    print(pearson)
    print("===========")

    return pearson

In [4]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.AddFlanks("", FromelDataset.RIGHT_FLANK),  # this is original parameters for human_legnet
        t.RightCrop(245, 270),  # this is using for shifting
        t.LeftCrop(245, 245),
        t.ReverseComplement(0.5),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.LeftCrop(245, 245),
        t.ReverseComplement(0),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)

forward_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.LeftCrop(245, 245),
        t.ReverseComplement(0),
        t.AddFeatureChannels(["batch"]),
        t.Seq2Tensor(),
    ]
)

reverse_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.LeftCrop(245, 245),
        t.ReverseComplement(1),
        t.AddFeatureChannels(["batch"]),
        t.Seq2Tensor(),
    ]
)

In [5]:
train_dataset = FromelDataset(split="train", transform=train_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")
val_dataset = FromelDataset(split="val", transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")

train_dl  = DataLoader(train_dataset, batch_size=1024, num_workers=16, shuffle=True)
val_dl  = DataLoader(val_dataset, batch_size=1024, num_workers=16, shuffle=False)

in_channels = len(train_dataset[0][0])
out_channels = len(train_dataset[0][1])

In [6]:
model = HumanLegNet(
        in_ch=in_channels,
        output_dim=out_channels,
        stem_ch=64,
        stem_ks=11,
        ef_ks=9,
        ef_block_sizes=[80, 96, 112, 128],
        pool_sizes=[2, 2, 2, 2],
        resize_factor=4,
)
model.apply(initialize_weights)
        
seq_model = LitModel_Fromel(
    model=model, 
    activity_columns=FromelDataset.HSPC_TARGETS,
    loss=MaskedMSE(),
    weight_decay=2e-1,
    lr=0.005, 
    print_each=20,
)

In [7]:
trainer = L.Trainer(
            accelerator="gpu",
            devices=[0],
            max_epochs=5,
            gradient_clip_val=1,
            precision="16-mixed",
            enable_progress_bar=True,
            num_sanity_val_steps=0,
        )
trainer.fit(seq_model, 
            train_dataloaders=train_dl,
            val_dataloaders=val_dl)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (45) is smaller than the lo

Epoch 0: 100%|██████████| 45/45 [00:04<00:00,  9.17it/s, v_num=110, train_loss_step=0.103, val_loss=0.112, val_State_1M_pearson=nan.0, val_State_2D_pearson=nan.0, val_State_3E_pearson=nan.0, val_State_4M_pearson=nan.0, val_State_5M_pearson=nan.0, val_State_6N_pearson=nan.0, val_State_7M_pearson=nan.0, val_pearson=nan.0, train_loss_epoch=0.109]

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32). Setting the correlation coefficient to nan.
  warnings.warn(*args, **kwargs)
Metric val_loss improved. New best score: 0.112


Epoch 2: 100%|██████████| 45/45 [00:05<00:00,  7.67it/s, v_num=110, train_loss_step=0.104, val_loss=0.0905, val_State_1M_pearson=nan.0, val_State_2D_pearson=nan.0, val_State_3E_pearson=nan.0, val_State_4M_pearson=nan.0, val_State_5M_pearson=nan.0, val_State_6N_pearson=nan.0, val_State_7M_pearson=nan.0, val_pearson=nan.0, train_loss_epoch=0.0931]

Metric val_loss improved by 0.022 >= min_delta = 0.0. New best score: 0.090


Epoch 3: 100%|██████████| 45/45 [00:04<00:00,  9.02it/s, v_num=110, train_loss_step=0.0853, val_loss=0.0887, val_State_1M_pearson=nan.0, val_State_2D_pearson=nan.0, val_State_3E_pearson=nan.0, val_State_4M_pearson=nan.0, val_State_5M_pearson=nan.0, val_State_6N_pearson=nan.0, val_State_7M_pearson=nan.0, val_pearson=nan.0, train_loss_epoch=0.0868]

Metric val_loss improved by 0.002 >= min_delta = 0.0. New best score: 0.089


Epoch 4: 100%|██████████| 45/45 [00:04<00:00,  9.15it/s, v_num=110, train_loss_step=0.0857, val_loss=0.0842, val_State_1M_pearson=nan.0, val_State_2D_pearson=nan.0, val_State_3E_pearson=nan.0, val_State_4M_pearson=nan.0, val_State_5M_pearson=nan.0, val_State_6N_pearson=nan.0, val_State_7M_pearson=nan.0, val_pearson=nan.0, train_loss_epoch=0.083] 

Metric val_loss improved by 0.004 >= min_delta = 0.0. New best score: 0.084
`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 45/45 [00:05<00:00,  8.91it/s, v_num=110, train_loss_step=0.0857, val_loss=0.0842, val_State_1M_pearson=nan.0, val_State_2D_pearson=nan.0, val_State_3E_pearson=nan.0, val_State_4M_pearson=nan.0, val_State_5M_pearson=nan.0, val_State_6N_pearson=nan.0, val_State_7M_pearson=nan.0, val_pearson=nan.0, train_loss_epoch=0.083]


In [8]:
test_dataset_forw = FromelDataset(split='test', transform=forward_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dataset_rev = FromelDataset(split='test', transform=reverse_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")

test_forw  = DataLoader(test_dataset_forw, batch_size=1024, num_workers=64, shuffle=False)
test_rev  = DataLoader(test_dataset_rev, batch_size=1024, num_workers=64, shuffle=False)

evaluate_with_reverse_complement(
            test_forw,
            test_rev,
            trainer,
            seq_model,
            FromelDataset.HSPC_TARGETS,
            num_outputs=out_channels,
        )

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 43.98it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 46.34it/s]
['State_1M', 'State_2D', 'State_3E', 'State_4M', 'State_5M', 'State_6N', 'State_7M']  Pearson correlation
tensor([0.5904, 0.5439, 0.4944, 0.6286, 0.5714, 0.4613, 0.4944])


tensor([0.5904, 0.5439, 0.4944, 0.6286, 0.5714, 0.4613, 0.4944])

In [9]:
test_dataset_forw = FromelDataset(split='genome', transform=forward_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dataset_rev = FromelDataset(split='genome', transform=reverse_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")

test_forw  = DataLoader(test_dataset_forw, batch_size=1024, num_workers=64, shuffle=False)
test_rev  = DataLoader(test_dataset_rev, batch_size=1024, num_workers=64, shuffle=False)

evaluate_with_reverse_complement(
            test_forw,
            test_rev,
            trainer,
            seq_model,
            FromelDataset.HSPC_TARGETS,
            num_outputs=out_channels,
        )

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 26.97it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 64.84it/s]
['State_1M', 'State_2D', 'State_3E', 'State_4M', 'State_5M', 'State_6N', 'State_7M']  Pearson correlation
tensor([-0.0492, -0.0190, -0.0231, -0.0316,     nan,  0.0534, -0.0472])


tensor([-0.0492, -0.0190, -0.0231, -0.0316,     nan,  0.0534, -0.0472])